# DAG Parameterization — Generative LGBN

This notebook implements a **generative classifier** using class-conditional Linear Gaussian Bayesian Networks.

Instead of treating `Outcome` as a Gaussian variable (the original LGBN approach, which misspecifies the binary outcome), this approach:
1. Fits separate LGBNs over the **feature sub-network only** — one for `Outcome=0`, one for `Outcome=1`
2. Applies Bayes rule: `P(Outcome=1 | X) ∝ P(X | Outcome=1) · P(Outcome=1)`
3. Computes likelihoods **node-by-node** using each CPD's linear parameters, avoiding the singular full covariance matrix issue at high feature counts

Results are structured identically to `DAG Parameterization.ipynb` for direct comparison.

In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import re
import glob
from scipy.stats import norm

from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.calibration import calibration_curve

from pgmpy.base import DAG
from pgmpy.models import LinearGaussianBayesianNetwork

from sklearn.metrics import roc_auc_score, average_precision_score
from pycalib.metrics import ECE

import xgboost
import shap

from MLstatkit import Bootstrapping, Delong_test, Permutation_test
from itertools import combinations

/Users/eddie/.pixi-envs/dag-parameterization-2248481784431669736/envs/default/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data preparation

In [2]:
X_train = pd.read_csv('../../Data Pre-processing/X_train_c12_w48_imp.csv', index_col=0)
X_test  = pd.read_csv('../../Data Pre-processing/X_test_c12_w48_imp.csv',  index_col=0)

print(X_train.shape, X_test.shape)

X_train = X_train.loc[:, X_train.var() >= 0.01]
X_test  = X_test.filter(items=X_train.columns)
print(X_train.shape, X_test.shape)

X_train_imp = pd.read_csv('../../Data Pre-processing/X_train_c12_w48_mice.csv', index_col=0)
X_test_imp  = pd.read_csv('../../Data Pre-processing/X_test_c12_w48_mice.csv',  index_col=0)

y_train = pd.read_csv('../../Data Pre-processing/y_train_c12_w48_imp.csv', index_col=[0,1]).groupby('uid').max()
y_test  = pd.read_csv('../../Data Pre-processing/y_test_c12_w48_imp.csv',  index_col=[0,1]).groupby('uid').max()

correlation_threshold = 0.3
X_train_corr = X_train_imp.loc[:, X_train_imp.corrwith(y_train['Outcome']).abs() >= correlation_threshold]
print(X_train_corr.shape)

(13054, 990) (3895, 990)
(13054, 811) (3895, 811)


(13054, 113)


## Load DAGs

In [3]:
adjacency_files = glob.glob("../DAGs/*_adjacency.csv")

dags = {}
dags['Control']     = nx.from_edgelist([(col, 'Outcome') for col in X_train_imp.columns.str.replace(r'(_.+)?$', '', regex=True).unique().tolist() if col != 'Outcome'], create_using=nx.DiGraph())
dags['Correlation'] = nx.from_edgelist([(col, 'Outcome') for col in X_train_corr.columns.str.replace(r'(_.+)?$', '', regex=True).unique().tolist() if col != 'Outcome'], create_using=nx.DiGraph())

for file in adjacency_files:
    dag_name = re.search(r'../DAGs/(.*)_adjacency.csv', file).group(1)
    dag_name = dag_name.replace('x', '$\\cap$')
    dag_name = dag_name.replace(' + ', ' $\\cup$ ')
    dags[dag_name] = nx.DiGraph(pd.read_csv(file, index_col=0))

dags.pop('Golem $\\cap$ PCMB')
list(dags.keys())

['Control',
 'Correlation',
 'Clinician Consensus $\\cup$ Golem',
 'Simplified Clinician Consensus $\\cup$ Simplified PCMB',
 'Simplified Clinician Consensus $\\cup$ Simplified Golem',
 'Clinician Consensus $\\cup$ PCMB',
 'Simplified Golem $\\cup$ Simplified PCMB',
 'Clinician Consensus',
 'Simplified Clinician Consensus',
 'Clinician Consensus $\\cap$ PCMB',
 'Golem',
 'PCMB',
 'Simplified Clinician Consensus $\\cap$ Simplified PCMB',
 'Simplified PCMB',
 'Clinician Consensus $\\cap$ Golem',
 'Simplified Golem',
 'Golem $\\cup$ PCMB',
 'Simplified Golem $\\cap$ Simplified PCMB',
 'Simplified Clinician Consensus $\\cap$ Simplified Golem']

## Generative LGBN Classifier

### Design
- `Outcome` is **not** a node in either LGBN — it is the class label.
- Two separate LGBNs are fit: one on `Outcome=0` samples, one on `Outcome=1` samples.
- Prediction uses the node-by-node factored log-likelihood to avoid inverting a potentially singular full covariance matrix.
- `predict_proba` is sklearn-compatible so it drops into the existing `performance_report` function unchanged.

In [4]:
def lgbn_log_likelihood(model, data):
    """Sum of node-wise conditional log-likelihoods over all nodes in the LGBN.

    LinearGaussianCPD stores:
      cpd.beta     — array where beta[0] is the intercept and beta[1:] are
                     coefficients for cpd.evidence (in matching order)
      cpd.std      — standard deviation of the node's Gaussian noise
      cpd.evidence — list of parent variable names
    """
    log_lik = np.zeros(len(data))
    for node in model.nodes():
        cpd = model.get_cpds(node)
        parents = cpd.evidence if cpd.evidence else []
        if parents:
            mu = cpd.beta[0] + data[parents].values @ cpd.beta[1:]
        else:
            mu = np.full(len(data), cpd.beta[0])
        log_lik += norm.logpdf(data[node].values, loc=mu, scale=cpd.std)
    return log_lik


class GenerativeLGBN:
    """
    Generative classifier using class-conditional Linear Gaussian Bayesian Networks.

    Fits P(X | class) for each class via a separate LGBN over the feature
    sub-network, then computes P(class | X) via Bayes rule.

    Features with no edges to other features are added as isolated nodes so
    they are still modeled as marginal Gaussians — this handles star DAGs
    (e.g. Control) where all edges point to Outcome and feature_edges is empty.

    Parameters
    ----------
    feature_edges : list of (str, str)
        Directed edges among features only — must not include 'Outcome'.
    """

    def __init__(self, feature_edges):
        self.feature_edges = feature_edges

    def fit(self, X, y):
        y = np.asarray(y)
        self.classes_ = np.array([0, 1])
        self.log_prior_ = np.array([
            np.log((y == 0).mean()),
            np.log((y == 1).mean()),
        ])
        self.lgbn_neg_ = LinearGaussianBayesianNetwork(self.feature_edges)
        self.lgbn_pos_ = LinearGaussianBayesianNetwork(self.feature_edges)
        # Ensure every feature column is a node, even if it has no edges.
        # Without this, star-topology DAGs (all edges go to Outcome) produce
        # an empty graph and return constant log-likelihoods for all samples.
        for col in X.columns:
            if col not in self.lgbn_neg_.nodes():
                self.lgbn_neg_.add_node(col)
            if col not in self.lgbn_pos_.nodes():
                self.lgbn_pos_.add_node(col)
        self.lgbn_neg_.fit(X[y == 0].reset_index(drop=True))
        self.lgbn_pos_.fit(X[y == 1].reset_index(drop=True))
        return self

    def predict_proba(self, X):
        X = X.reset_index(drop=True)
        log_lik_pos = lgbn_log_likelihood(self.lgbn_pos_, X) + self.log_prior_[1]
        log_lik_neg = lgbn_log_likelihood(self.lgbn_neg_, X) + self.log_prior_[0]
        log_sum = np.logaddexp(log_lik_pos, log_lik_neg)
        p1 = np.clip(np.exp(log_lik_pos - log_sum), 0.0, 1.0)
        return np.column_stack([1 - p1, p1])

## Evaluation functions

In [5]:
def performance_report(X, y, model):
    y_prob = model.predict_proba(X)[:, 1]

    n_bootstraps = 1000
    ap,       ap_lo,        ap_hi       = Bootstrapping(y.values, y_prob, metric_str='average_precision', n_bootstraps=n_bootstraps, confidence_level=0.95, random_state=4242)
    auroc,    auroc_lo,     auroc_hi    = Bootstrapping(y.values, y_prob, metric_str='roc_auc',           n_bootstraps=n_bootstraps, confidence_level=0.95, random_state=4242)
    prec,     prec_lo,      prec_hi     = Bootstrapping(y.values, y_prob, metric_str='precision',         n_bootstraps=n_bootstraps, confidence_level=0.95, random_state=4242)
    rec,      rec_lo,       rec_hi      = Bootstrapping(y.values, y_prob, metric_str='recall',            n_bootstraps=n_bootstraps, confidence_level=0.95, random_state=4242)
    f1,       f1_lo,        f1_hi       = Bootstrapping(y.values, y_prob, metric_str='f1',                n_bootstraps=n_bootstraps, confidence_level=0.95, random_state=4242)
    acc,      acc_lo,       acc_hi      = Bootstrapping(y.values, y_prob, metric_str='accuracy',          n_bootstraps=n_bootstraps, confidence_level=0.95, random_state=4242)
    ece = ECE(y.values, y_prob.reshape(-1, 1), bins=50)

    return {
        'Average Precision': f"{ap:.2f} ({ap_lo:.2f}, {ap_hi:.2f})",
        'AUROC':             f"{auroc:.2f} ({auroc_lo:.2f}, {auroc_hi:.2f})",
        'Precision':         f"{prec:.2f} ({prec_lo:.2f}, {prec_hi:.2f})",
        'Recall':            f"{rec:.2f} ({rec_lo:.2f}, {rec_hi:.2f})",
        'F1 Score':          f"{f1:.2f} ({f1_lo:.2f}, {f1_hi:.2f})",
        'Accuracy':          f"{acc:.2f} ({acc_lo:.2f}, {acc_hi:.2f})",
        'ECE':               f"{ece:.3f}",
    }


def compare_models(y, y_prob1, y_prob2):
    z, p = Delong_test(y.values, y_prob1, y_prob2, return_ci=False, return_auc=False)
    result = Permutation_test(y.values, y_prob1, y_prob2, metric_str='average_precision')
    return (z, p), result

## Model training

In [6]:
drugs = [
    'Midazolam', 'Fentanyl', 'Olanzapine', 'Haloperidol',
    'Dexmedetomidine', 'Lorazepam', 'Morphine', 'Hydromorphone',
    'Dopamine', 'Cisatracurium', 'Epinephrine', 'Norepinephrine',
    'Milrinone', 'Dobutamine',
]
intervention_nodes = [
    'CRRT Therapy Type', 'ECMO Type', 'Ventilated',
    'Pupillary Reaction', 'Peds Coma Score', 'Endotracheal tube',
]


def train_models(remove_drugs=False, remove_interventions=False):
    results = []
    calibrations = {}
    model_preds = {}

    for dag_name, dag in dags.items():

        print(f"Processing {dag_name} — {dag.number_of_nodes()} nodes, {dag.number_of_edges()} edges")

        nodes_in_dag = list(dag.nodes())

        if remove_drugs:
            nodes_in_dag = [n for n in nodes_in_dag if not any(d in n for d in drugs)]
        if remove_interventions:
            nodes_in_dag = [n for n in nodes_in_dag if n not in intervention_nodes]

        # --- feature expansion ---
        if dag_name == 'Correlation':
            source = X_train_corr
        else:
            source = X_train_imp

        if 'Simplified' in dag_name or dag_name in ['Control', 'Correlation']:
            # Node names are high-level biomarker names; expand to feature columns
            features_in_dag = [
                col for col in source.columns
                if any(re.search(rf'^{re.escape(node)}(_.+)?$', col) for node in nodes_in_dag)
            ]
            train_dag = nx.DiGraph()
            for node in nodes_in_dag:
                ends = list(dag.successors(node))
                from_features = [c for c in features_in_dag + ['Outcome'] if re.search(rf'^{re.escape(node)}(_.+)?$', c)]
                to_features   = [c for c in features_in_dag + ['Outcome'] if any(re.search(rf'^{re.escape(e)}(_.+)?$', c) for e in ends)]
                for f in from_features:
                    for t in to_features:
                        train_dag.add_edge(f, t)
        else:
            # Node names are already feature column names
            features_in_dag = X_train_imp.filter(items=nodes_in_dag).columns.tolist()
            train_dag = dag.subgraph(features_in_dag + ['Outcome']).copy()

        n_biomarkers = len(set(re.sub(r'(_.+)?$', '', c) for c in features_in_dag))

        # Feature-only edges — Outcome stripped from the LGBN structure
        feature_edges = [
            (u, v) for u, v in train_dag.edges()
            if u != 'Outcome' and v != 'Outcome'
        ]

        X_tr = X_train_imp[features_in_dag] if dag_name != 'Correlation' else X_train_corr[features_in_dag]
        X_te = X_test_imp[features_in_dag]  if dag_name != 'Correlation' else X_test_imp.filter(features_in_dag)

        # --- Generative LGBN ---
        gen_lgbn = GenerativeLGBN(feature_edges)
        gen_lgbn.fit(X_tr, y_train['Outcome'].values)

        row = {'Model': 'Generative LGBN', 'DAG': dag_name}
        row.update(performance_report(X_te, y_test.Outcome, gen_lgbn))
        row['# Features']   = len(features_in_dag)
        row['# Biomarkers'] = n_biomarkers
        results.append(row)

        # --- XGBoost ---
        xgb = xgboost.XGBClassifier(
            objective='binary:logistic', random_state=42, eval_metric='aucpr'
        )
        X_tr_xgb = X_train[features_in_dag] if dag_name != 'Correlation' else X_train.filter(features_in_dag)
        X_te_xgb = X_test[features_in_dag]  if dag_name != 'Correlation' else X_test.filter(features_in_dag)
        xgb.fit(X_tr_xgb, y_train.Outcome)

        row_xgb = {'Model': 'XGB', 'DAG': dag_name}
        row_xgb.update(performance_report(X_te_xgb, y_test.Outcome, xgb))
        row_xgb['# Features']   = len(features_in_dag)
        row_xgb['# Biomarkers'] = n_biomarkers
        results.append(row_xgb)

        model_preds[dag_name] = {
            'Generative LGBN': gen_lgbn.predict_proba(X_te)[:, 1],
            'XGB':             xgb.predict_proba(X_te_xgb)[:, 1],
        }

        calibrations[dag_name] = calibration_curve(
            y_test.Outcome, xgb.predict_proba(X_te_xgb)[:, 1], n_bins=50
        )

    return results, calibrations, model_preds

## Experiment 1: All features

In [7]:
results1, calibrations1, model_preds1 = train_models(remove_drugs=False, remove_interventions=False)

Processing Control — 46 nodes, 45 edges


Bootstrapping average_precision:   0%|          | 0/1000 [00:00<?, ?it/s]

Bootstrapping average_precision:  14%|█▎        | 136/1000 [00:00<00:00, 1356.75it/s]

Bootstrapping average_precision:  28%|██▊       | 275/1000 [00:00<00:00, 1375.57it/s]

Bootstrapping average_precision:  41%|████▏     | 414/1000 [00:00<00:00, 1379.84it/s]

...

Bootstrapping f1: 100%|██████████| 1000/1000 [00:00<00:00, 1210.10it/s]

Bootstrapping accuracy:   0%|          | 0/1000 [00:00<?, ?it/s]

Bootstrapping accuracy:  35%|███▌      | 354/1000 [00:00<00:00, 3532.61it/s]

Bootstrapping accuracy:  71%|███████   | 710/1000 [00:00<00:00, 3547.37it/s]

Bootstrapping accuracy: 100%|██████████| 1000/1000 [00:00<00:00, 3548.17it/s]

In [8]:
df1 = pd.DataFrame(results1)
df1.to_csv('Generative LGBN - Biomarker Selection.csv', index=False)
df1

,Model,DAG,Average Precision,AUROC,Precision,Recall,F1 Score,Accuracy,ECE,# Features,# Biomarkers
0,Generative LGBN,Control,"0.48 (0.44, 0.52)","0.84 (0.82, 0.86)","0.77 (0.74, 0.79)","0.78 (0.75, 0.80)","0.77 (0.75, 0.79)","0.91 (0.90, 0.92)",0.116,811,45
1,XGB,Control,"0.81 (0.78, 0.84)","0.93 (0.91, 0.95)","0.94 (0.92, 0.95)","0.80 (0.78, 0.83)","0.85 (0.83, 0.87)","0.95 (0.94, 0.96)",0.089,811,45
2,Generative LGBN,Correlation,"0.51 (0.47, 0.56)","0.80 (0.78, 0.83)","0.81 (0.78, 0.83)","0.74 (0.72, 0.77)","0.77 (0.75, 0.79)","0.92 (0.91, 0.93)",0.085,113,37
3,XGB,Correlation,"0.75 (0.71, 0.78)","0.90 (0.88, 0.91)","0.90 (0.88, 0.92)","0.78 (0.76, 0.81)","0.83 (0.81, 0.85)","0.94 (0.94, 0.95)",0.108,113,37
4,Generative LGBN,Clinician Consensus $\cup$ Golem,"0.53 (0.49, 0.58)","0.84 (0.82, 0.86)","0.77 (0.75, 0.79)","0.77 (0.74, 0.79)","0.77 (0.75, 0.79)","0.91 (0.90, 0.92)",0.108,212,22
5,XGB,Clinician Consensus $\cup$ Golem,"0.79 (0.75, 0.82)","0.92 (0.90, 0.93)","0.91 (0.89, 0.93)","0.80 (0.78, 0.82)","0.85 (0.83, 0.87)","0.95 (0.94, 0.95)",0.096,212,22
6,Generative LGBN,Simplified Clinician Consensus $\cup$ Simplifi...,"0.53 (0.49, 0.58)","0.87 (0.85, 0.89)","0.79 (0.77, 0.81)","0.79 (0.76, 0.81)","0.79 (0.77, 0.81)","0.92 (0.91, 0.93)",0.110,315,18
7,XGB,Simplified Clinician Consensus $\cup$ Simplifi...,"0.79 (0.75, 0.82)","0.91 (0.89, 0.93)","0.92 (0.90, 0.93)","0.80 (0.78, 0.83)","0.85 (0.83, 0.87)","0.95 (0.94, 0.96)",0.094,315,18
8,Generative LGBN,Simplified Clinician Consensus $\cup$ Simplifi...,"0.51 (0.46, 0.55)","0.84 (0.82, 0.87)","0.77 (0.75, 0.80)","0.77 (0.75, 0.79)","0.77 (0.75, 0.79)","0.91 (0.90, 0.92)",0.109,389,22
9,XGB,Simplified Clinician Consensus $\cup$ Simplifi...,"0.78 (0.75, 0.82)","0.91 (0.89, 0.93)","0.92 (0.90, 0.94)","0.79 (0.77, 0.82)","0.84 (0.82, 0.86)","0.95 (0.94, 0.95)",0.092,389,22


In [9]:
# DeLong / permutation comparisons: Generative LGBN vs XGBoost within each DAG
aurocs1, auprcs1 = [], []
for dag_name, preds in model_preds1.items():
    p_lgbn = preds['Generative LGBN']
    p_xgb  = preds['XGB']
    (z, p_delong), perm = compare_models(y_test.Outcome, p_lgbn, p_xgb)
    aurocs1.append({'DAG': dag_name, 'Z-score': f"{z:.4f}", 'P-value': f"{p_delong:.3e}"})
    auprcs1.append({'DAG': dag_name,
                    'Avg Precision LGBN': f"{perm[0]:.4f}",
                    'Avg Precision XGB':  f"{perm[1]:.4f}",
                    'P-value': f"{perm[2]:.3e}"})

aurocs1_df = pd.DataFrame(aurocs1)
auprcs1_df = pd.DataFrame(auprcs1)
aurocs1_df.to_csv('Generative LGBN - delong_biomarker_selection_auroc.csv', index=False)
auprcs1_df.to_csv('Generative LGBN - delong_biomarker_selection_auprc.csv', index=False)
aurocs1_df

Computing average_precision Permutation Test p-value:   0%|          | 0/1000 [00:00<?, ?it/s]

Computing average_precision Permutation Test p-value:   7%|▋         | 70/1000 [00:00<00:01, 698.72it/s]

Computing average_precision Permutation Test p-value:  14%|█▍        | 142/1000 [00:00<00:01, 710.79it/s]

Computing average_precision Permutation Test p-value:  22%|██▏       | 215/1000 [00:00<00:01, 718.17it/s]

Computing average_precision Permutation Test p-value:  29%|██▉       | 288/1000 [00:00<00:00, 721.77it/s]

...

Computing average_precision Permutation Test p-value:  86%|████████▌ | 855/1000 [00:01<00:00, 708.90it/s]

Computing average_precision Permutation Test p-value:  93%|█████████▎| 926/1000 [00:01<00:00, 709.17it/s]

Computing average_precision Permutation Test p-value: 100%|█████████▉| 999/1000 [00:01<00:00, 713.49it/s]

Computing average_precision Permutation Test p-value: 100%|██████████| 1000/1000 [00:01<00:00, 705.09it/s]

,DAG,Z-score,P-value
0,Control,8.8821,6.562e-19
1,Correlation,7.7685,7.941e-15
2,Clinician Consensus $\cup$ Golem,7.2008,5.985e-13
3,Simplified Clinician Consensus $\cup$ Simplifi...,4.5700,4.878e-06
4,Simplified Clinician Consensus $\cup$ Simplifi...,6.1230,9.184e-10
5,Clinician Consensus $\cup$ PCMB,5.4507,5.017e-08
6,Simplified Golem $\cup$ Simplified PCMB,5.3593,8.352e-08
7,Clinician Consensus,5.7698,7.939e-09
8,Simplified Clinician Consensus,6.1615,7.205e-10
9,Clinician Consensus $\cap$ PCMB,2.8826,3.944e-03


## Experiment 2: No drugs

In [10]:
results2, calibrations2, model_preds2 = train_models(remove_drugs=True, remove_interventions=False)

Processing Control — 46 nodes, 45 edges


Bootstrapping average_precision:   0%|          | 0/1000 [00:00<?, ?it/s]

Bootstrapping average_precision:  13%|█▎        | 134/1000 [00:00<00:00, 1332.51it/s]

Bootstrapping average_precision:  27%|██▋       | 268/1000 [00:00<00:00, 1324.46it/s]

Bootstrapping average_precision:  40%|████      | 403/1000 [00:00<00:00, 1335.00it/s]

...

Bootstrapping f1: 100%|██████████| 1000/1000 [00:00<00:00, 1210.83it/s]

Bootstrapping accuracy:   0%|          | 0/1000 [00:00<?, ?it/s]

Bootstrapping accuracy:  36%|███▌      | 356/1000 [00:00<00:00, 3555.41it/s]

Bootstrapping accuracy:  71%|███████▏  | 714/1000 [00:00<00:00, 3565.90it/s]

Bootstrapping accuracy: 100%|██████████| 1000/1000 [00:00<00:00, 3560.71it/s]

In [11]:
df2 = pd.DataFrame(results2)
df2.to_csv('Generative LGBN - No Drugs.csv', index=False)
df2

,Model,DAG,Average Precision,AUROC,Precision,Recall,F1 Score,Accuracy,ECE,# Features,# Biomarkers
0,Generative LGBN,Control,"0.48 (0.44, 0.53)","0.84 (0.82, 0.86)","0.76 (0.74, 0.78)","0.78 (0.76, 0.80)","0.77 (0.75, 0.79)","0.91 (0.90, 0.92)",0.121,624,34
1,XGB,Control,"0.80 (0.77, 0.83)","0.92 (0.91, 0.94)","0.92 (0.90, 0.94)","0.80 (0.78, 0.82)","0.85 (0.83, 0.87)","0.95 (0.94, 0.96)",0.090,624,34
2,Generative LGBN,Correlation,"0.54 (0.49, 0.59)","0.81 (0.78, 0.83)","0.82 (0.80, 0.85)","0.74 (0.72, 0.77)","0.77 (0.75, 0.79)","0.92 (0.91, 0.93)",0.080,70,26
3,XGB,Correlation,"0.75 (0.71, 0.79)","0.89 (0.87, 0.91)","0.90 (0.88, 0.92)","0.78 (0.76, 0.80)","0.82 (0.80, 0.85)","0.94 (0.93, 0.95)",0.111,70,26
4,Generative LGBN,Clinician Consensus $\cup$ Golem,"0.54 (0.49, 0.58)","0.84 (0.82, 0.87)","0.77 (0.75, 0.80)","0.77 (0.75, 0.79)","0.77 (0.75, 0.79)","0.91 (0.90, 0.92)",0.108,134,15
5,XGB,Clinician Consensus $\cup$ Golem,"0.77 (0.73, 0.80)","0.90 (0.88, 0.92)","0.91 (0.89, 0.93)","0.80 (0.77, 0.82)","0.84 (0.82, 0.86)","0.95 (0.94, 0.95)",0.100,134,15
6,Generative LGBN,Simplified Clinician Consensus $\cup$ Simplifi...,"0.52 (0.47, 0.56)","0.85 (0.83, 0.87)","0.77 (0.75, 0.79)","0.77 (0.74, 0.79)","0.77 (0.75, 0.79)","0.91 (0.90, 0.92)",0.110,198,11
7,XGB,Simplified Clinician Consensus $\cup$ Simplifi...,"0.77 (0.74, 0.81)","0.91 (0.89, 0.93)","0.91 (0.89, 0.93)","0.79 (0.77, 0.81)","0.84 (0.81, 0.86)","0.94 (0.94, 0.95)",0.100,198,11
8,Generative LGBN,Simplified Clinician Consensus $\cup$ Simplifi...,"0.50 (0.46, 0.56)","0.85 (0.82, 0.87)","0.76 (0.74, 0.79)","0.77 (0.75, 0.79)","0.77 (0.75, 0.79)","0.91 (0.90, 0.92)",0.114,271,15
9,XGB,Simplified Clinician Consensus $\cup$ Simplifi...,"0.77 (0.74, 0.81)","0.91 (0.89, 0.93)","0.91 (0.89, 0.92)","0.80 (0.78, 0.82)","0.84 (0.82, 0.86)","0.95 (0.94, 0.95)",0.099,271,15


In [12]:
aurocs2, auprcs2 = [], []
for dag_name, preds in model_preds2.items():
    (z, p_delong), perm = compare_models(y_test.Outcome, preds['Generative LGBN'], preds['XGB'])
    aurocs2.append({'DAG': dag_name, 'Z-score': f"{z:.4f}", 'P-value': f"{p_delong:.3e}"})
    auprcs2.append({'DAG': dag_name,
                    'Avg Precision LGBN': f"{perm[0]:.4f}",
                    'Avg Precision XGB':  f"{perm[1]:.4f}",
                    'P-value': f"{perm[2]:.3e}"})

pd.DataFrame(aurocs2).to_csv('Generative LGBN - delong_no_drugs_auroc.csv', index=False)
pd.DataFrame(auprcs2).to_csv('Generative LGBN - delong_no_drugs_auprc.csv', index=False)
pd.DataFrame(aurocs2)

Computing average_precision Permutation Test p-value:   0%|          | 0/1000 [00:00<?, ?it/s]

Computing average_precision Permutation Test p-value:   7%|▋         | 73/1000 [00:00<00:01, 729.45it/s]

Computing average_precision Permutation Test p-value:  15%|█▍        | 146/1000 [00:00<00:01, 727.35it/s]

Computing average_precision Permutation Test p-value:  22%|██▏       | 219/1000 [00:00<00:01, 728.00it/s]

Computing average_precision Permutation Test p-value:  29%|██▉       | 292/1000 [00:00<00:00, 724.82it/s]

...

Computing average_precision Permutation Test p-value:  81%|████████  | 808/1000 [00:01<00:00, 728.74it/s]

Computing average_precision Permutation Test p-value:  88%|████████▊ | 881/1000 [00:01<00:00, 727.69it/s]

Computing average_precision Permutation Test p-value:  95%|█████████▌| 954/1000 [00:01<00:00, 728.20it/s]

Computing average_precision Permutation Test p-value: 100%|██████████| 1000/1000 [00:01<00:00, 728.85it/s]

,DAG,Z-score,P-value
0,Control,7.9325,2.149e-15
1,Correlation,6.7043,2.023e-11
2,Clinician Consensus $\cup$ Golem,5.6721,1.410e-08
3,Simplified Clinician Consensus $\cup$ Simplifi...,5.7669,8.074e-09
4,Simplified Clinician Consensus $\cup$ Simplifi...,6.0204,1.740e-09
5,Clinician Consensus $\cup$ PCMB,5.5715,2.526e-08
6,Simplified Golem $\cup$ Simplified PCMB,6.3830,1.737e-10
7,Clinician Consensus,5.6888,1.280e-08
8,Simplified Clinician Consensus,5.0814,3.747e-07
9,Clinician Consensus $\cap$ PCMB,3.5034,4.593e-04


## Experiment 3: No drugs or interventions

In [13]:
results3, calibrations3, model_preds3 = train_models(remove_drugs=True, remove_interventions=True)

Processing Control — 46 nodes, 45 edges


Bootstrapping average_precision:   0%|          | 0/1000 [00:00<?, ?it/s]

Bootstrapping average_precision:  14%|█▎        | 137/1000 [00:00<00:00, 1366.55it/s]

Bootstrapping average_precision:  28%|██▊       | 275/1000 [00:00<00:00, 1373.45it/s]

Bootstrapping average_precision:  42%|████▏     | 415/1000 [00:00<00:00, 1381.32it/s]

...

Bootstrapping f1: 100%|██████████| 1000/1000 [00:00<00:00, 1211.10it/s]

Bootstrapping accuracy:   0%|          | 0/1000 [00:00<?, ?it/s]

Bootstrapping accuracy:  35%|███▌      | 354/1000 [00:00<00:00, 3534.63it/s]

Bootstrapping accuracy:  72%|███████▏  | 715/1000 [00:00<00:00, 3578.06it/s]

Bootstrapping accuracy: 100%|██████████| 1000/1000 [00:00<00:00, 3561.14it/s]

In [14]:
df3 = pd.DataFrame(results3)
df3.to_csv('Generative LGBN - No Drugs or Interventions.csv', index=False)
df3

,Model,DAG,Average Precision,AUROC,Precision,Recall,F1 Score,Accuracy,ECE,# Features,# Biomarkers
0,Generative LGBN,Control,"0.47 (0.43, 0.51)","0.84 (0.81, 0.86)","0.76 (0.73, 0.78)","0.78 (0.75, 0.80)","0.77 (0.74, 0.79)","0.90 (0.89, 0.91)",0.121,521,28
1,XGB,Control,"0.78 (0.75, 0.82)","0.92 (0.90, 0.93)","0.90 (0.88, 0.92)","0.80 (0.78, 0.82)","0.84 (0.82, 0.86)","0.95 (0.94, 0.95)",0.093,521,28
2,Generative LGBN,Correlation,"0.55 (0.50, 0.60)","0.81 (0.78, 0.83)","0.82 (0.80, 0.85)","0.74 (0.71, 0.76)","0.77 (0.75, 0.79)","0.92 (0.91, 0.93)",0.078,53,20
3,XGB,Correlation,"0.75 (0.71, 0.78)","0.89 (0.87, 0.91)","0.90 (0.88, 0.92)","0.78 (0.76, 0.80)","0.83 (0.80, 0.85)","0.94 (0.93, 0.95)",0.114,53,20
4,Generative LGBN,Clinician Consensus $\cup$ Golem,"0.54 (0.49, 0.58)","0.84 (0.82, 0.87)","0.77 (0.75, 0.80)","0.77 (0.75, 0.79)","0.77 (0.75, 0.79)","0.91 (0.90, 0.92)",0.108,134,15
5,XGB,Clinician Consensus $\cup$ Golem,"0.77 (0.73, 0.80)","0.90 (0.88, 0.92)","0.91 (0.89, 0.93)","0.80 (0.77, 0.82)","0.84 (0.82, 0.86)","0.95 (0.94, 0.95)",0.100,134,15
6,Generative LGBN,Simplified Clinician Consensus $\cup$ Simplifi...,"0.47 (0.42, 0.51)","0.83 (0.80, 0.85)","0.73 (0.71, 0.76)","0.74 (0.72, 0.77)","0.74 (0.72, 0.76)","0.89 (0.88, 0.90)",0.116,131,7
7,XGB,Simplified Clinician Consensus $\cup$ Simplifi...,"0.76 (0.73, 0.80)","0.91 (0.89, 0.92)","0.89 (0.87, 0.91)","0.79 (0.77, 0.81)","0.83 (0.81, 0.85)","0.94 (0.93, 0.95)",0.104,131,7
8,Generative LGBN,Simplified Clinician Consensus $\cup$ Simplifi...,"0.48 (0.43, 0.53)","0.83 (0.81, 0.86)","0.73 (0.71, 0.75)","0.76 (0.73, 0.78)","0.74 (0.72, 0.76)","0.89 (0.88, 0.90)",0.126,204,11
9,XGB,Simplified Clinician Consensus $\cup$ Simplifi...,"0.77 (0.74, 0.81)","0.91 (0.90, 0.93)","0.91 (0.89, 0.93)","0.79 (0.77, 0.81)","0.84 (0.82, 0.86)","0.95 (0.94, 0.95)",0.101,204,11


In [15]:
aurocs3, auprcs3 = [], []
for dag_name, preds in model_preds3.items():
    (z, p_delong), perm = compare_models(y_test.Outcome, preds['Generative LGBN'], preds['XGB'])
    aurocs3.append({'DAG': dag_name, 'Z-score': f"{z:.4f}", 'P-value': f"{p_delong:.3e}"})
    auprcs3.append({'DAG': dag_name,
                    'Avg Precision LGBN': f"{perm[0]:.4f}",
                    'Avg Precision XGB':  f"{perm[1]:.4f}",
                    'P-value': f"{perm[2]:.3e}"})

pd.DataFrame(aurocs3).to_csv('Generative LGBN - delong_only_vitals_labs_auroc.csv', index=False)
pd.DataFrame(auprcs3).to_csv('Generative LGBN - delong_only_vitals_labs_auprc.csv', index=False)
pd.DataFrame(aurocs3)

Computing average_precision Permutation Test p-value:   0%|          | 0/1000 [00:00<?, ?it/s]

Computing average_precision Permutation Test p-value:   7%|▋         | 74/1000 [00:00<00:01, 730.95it/s]

Computing average_precision Permutation Test p-value:  15%|█▍        | 148/1000 [00:00<00:01, 733.65it/s]

Computing average_precision Permutation Test p-value:  22%|██▏       | 222/1000 [00:00<00:01, 727.23it/s]

Computing average_precision Permutation Test p-value:  30%|██▉       | 295/1000 [00:00<00:00, 724.86it/s]

...

Computing average_precision Permutation Test p-value:  80%|████████  | 802/1000 [00:01<00:00, 723.42it/s]

Computing average_precision Permutation Test p-value:  88%|████████▊ | 876/1000 [00:01<00:00, 725.79it/s]

Computing average_precision Permutation Test p-value:  95%|█████████▍| 949/1000 [00:01<00:00, 726.33it/s]

Computing average_precision Permutation Test p-value: 100%|██████████| 1000/1000 [00:01<00:00, 723.42it/s]

,DAG,Z-score,P-value
0,Control,8.1156,4.836e-16
1,Correlation,7.1856,6.690e-13
2,Clinician Consensus $\cup$ Golem,5.6721,1.410e-08
3,Simplified Clinician Consensus $\cup$ Simplifi...,7.7323,1.056e-14
4,Simplified Clinician Consensus $\cup$ Simplifi...,7.4417,9.942e-14
5,Clinician Consensus $\cup$ PCMB,5.5715,2.526e-08
6,Simplified Golem $\cup$ Simplified PCMB,7.9977,1.268e-15
7,Clinician Consensus,5.6888,1.280e-08
8,Simplified Clinician Consensus,6.7090,1.959e-11
9,Clinician Consensus $\cap$ PCMB,3.5034,4.593e-04


## Summary: Generative LGBN vs XGBoost across all experiments

In [16]:
def extract_mean(s):
    """Extract the point estimate from a string like '0.76 (0.72, 0.80)'."""
    return float(str(s).split(' ')[0])


summary_rows = []
for experiment, df in [
    ('All Features',              pd.DataFrame(results1)),
    ('No Drugs',                  pd.DataFrame(results2)),
    ('No Drugs or Interventions', pd.DataFrame(results3)),
]:
    for model_name in ['Generative LGBN', 'XGB']:
        sub = df[df['Model'] == model_name].copy()
        sub['_ap'] = sub['Average Precision'].apply(extract_mean)
        best = sub.loc[sub['_ap'].idxmax()]
        summary_rows.append({
            'Experiment':         experiment,
            'Model':              model_name,
            'Best DAG':           best['DAG'],
            'Average Precision':  best['Average Precision'],
            'AUROC':              best['AUROC'],
            'ECE':                best['ECE'],
            '# Features':        best['# Features'],
        })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv('Generative LGBN - Summary.csv', index=False)
summary_df

,Experiment,Model,Best DAG,Average Precision,AUROC,ECE,# Features
0,All Features,Generative LGBN,Clinician Consensus $\cap$ PCMB,"0.59 (0.55, 0.64)","0.82 (0.79, 0.84)",0.078,20
1,All Features,XGB,Control,"0.81 (0.78, 0.84)","0.93 (0.91, 0.95)",0.089,811
2,No Drugs,Generative LGBN,Clinician Consensus $\cap$ PCMB,"0.57 (0.52, 0.61)","0.80 (0.77, 0.83)",0.098,14
3,No Drugs,XGB,Control,"0.80 (0.77, 0.83)","0.92 (0.91, 0.94)",0.090,624
4,No Drugs or Interventions,Generative LGBN,Clinician Consensus $\cap$ PCMB,"0.57 (0.52, 0.61)","0.80 (0.77, 0.83)",0.098,14
5,No Drugs or Interventions,XGB,Control,"0.78 (0.75, 0.82)","0.92 (0.90, 0.93)",0.093,521
